## Agentic Pattern Used - Evaluator-Optimizer

In [43]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import json
from IPython.display import Markdown, display

In [44]:
load_dotenv(override=True)

True

In [45]:
open_api_key = os.getenv("OPENAI_API_KEY")
google_api_key = os.getenv('GOOGLE_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

In [46]:
if open_api_key:
    print(f"the key starts with {open_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if google_api_key:
    print(f"the key starts with {google_api_key[:2]}")
else:
    print("gemini API Key not set")    

if groq_api_key:
   print(f"Groq API Key exists and begins {groq_api_key[:4]}")    
else:
    print("Groq API Key not set")

    
    

the key starts with sk-proj-
the key starts with AI
Groq API Key exists and begins gsk_


In [47]:
openai = OpenAI()

### I have attempted routing pattern in the below example. Depending on the current_query, the LLM would choose the appropriate format and call that function. Try to change the query from strings to int and see the change in output.

In [48]:
system_prompt="""You are a helpful assistant that can perform mathematical calculations.
Respond  with exactly one of the following formats:
1. FUNCTION_CALL: function_name|input
2. FINAL_ANSWER: [number]

where the function_name can be one of the following:
1. strings_to_chars_to_int(string) It takes a word as input, and returns the ASCII INT values of characters in the word as a list
2. int_list_to_exponential_sum(list) It takes a list of integers and returns the sum of exponentials of those integers
3. fibonacci_numbers(int) It takes an integer, like 6, and returns first 6 integers in a fibonacci series as a list.
DO NOT include multiple responses. Give ONE response at a time.
"""

current_query="""Calculate the sum of exponentials of word "INDIA"""

prompt = f"{system_prompt}\n\n Query: {current_query}"
response = openai.chat.completions.create(
    model="gpt-4.1-nano",
    messages=[{"role":"system", "content": prompt}])

print(response.choices[0].message.content)

FUNCTION_CALL: strings_to_chars_to_int|"INDIA"


In [49]:
import math

def strings_to_chars_to_int(string):
    return [ord(char) for char in string]

def int_list_to_exponential_sum(int_list):
    int_list = eval(int_list)
    return sum(math.exp(x) for x in int_list)

def fibonacci_numbers(n):
    if n <= 0:
        return []
    fib_sequence = [0, 1]
    for _ in range(2, n):
        fib_sequence.append(fib_sequence[-1] + fib_sequence[-2])
    return fib_sequence[:n]

In [50]:
response = response.choices[0].message.content
print(f"Response from model: {response}")

Response from model: FUNCTION_CALL: strings_to_chars_to_int|"INDIA"


In [51]:
_, function_info = response.split(":", 1)
_, function_info

('FUNCTION_CALL', ' strings_to_chars_to_int|"INDIA"')

In [52]:
func_name, params = [x.strip() for x in function_info.split("|", 1)]

func_name, params

('strings_to_chars_to_int', '"INDIA"')

In [53]:
def function_caller(func_name, params):
    """Simple function caller that maps function names to actual functions"""
    function_map = {
        "strings_to_chars_to_int": strings_to_chars_to_int,
        "int_list_to_exponential_sum": int_list_to_exponential_sum,
        "fibonacci_numbers": fibonacci_numbers
    }
    
    if func_name in function_map:
        return function_map[func_name](params)
    else:
        return f"Function {func_name} not found"

In [54]:
iteration_result = function_caller(func_name, params)
print(f"Result of function call: {iteration_result}")

Result of function call: [34, 73, 78, 68, 73, 65, 34]


In [55]:
system_prompt="""You are a helpful assistant that can perform mathematical calculations.
Respond  with exactly one of the following formats:
1. FUNCTION_CALL: function_name|input
2. FINAL_ANSWER: [number]

where the function_name can be one of the following:
1. strings_to_chars_to_int(string) It takes a word as input, and returns the ASCII INT values of characters in the word as a list
2. int_list_to_exponential_sum(list) It takes a list of integers and returns the sum of exponentials of those integers
3. fibonacci_numbers(int) It takes an integer, like 6, and returns first 6 integers in a fibonacci series as a list.
DO NOT include multiple responses. Give ONE response at a time.
"""

current_query="""Calculate the fibonacci_numbers of 3"""

prompt = f"{system_prompt}\n\n Query: {current_query}"
response = openai.chat.completions.create(
    model="gpt-4.1-nano",
    messages=[{"role":"system", "content": prompt}])

print(response.choices[0].message.content)

FINAL_ANSWER: [0, 1, 1]


In [56]:
judge_llm = f"""You are an evaluator. Your job is to check whether the response below follows the system prompt's rules AND is correct.

SYSTEM PROMPT BEING EVALUATED:
{system_prompt}

USER QUERY:
{current_query}

MODEL RESPONSE:
{response}

Evaluate against these criteria:

1. structured_output
   - Does the response match EXACTLY one of the two allowed formats (FUNCTION_CALL: name|input OR FINAL_ANSWER: [number])?
   - Is it parseable without extra text, explanations, or markdown around it?

2. correct_function_selected
   - Given the query, is this the RIGHT function to call (not just A valid function)?
   - Example: "Calculate the fibonacci_numbers of 3" must call fibonacci_numbers, not any other listed function.

3. input_format_valid
   - Does the input passed to the function match what that function expects (e.g., an integer for fibonacci_numbers, not a string)?

4. no_hallucinated_function
   - Does the function_name actually exist in the system prompt's list of 3 functions?
   - If FINAL_ANSWER was given instead of a function call, was that appropriate (i.e., was no tool actually needed)?

For each criterion, first write ONE short sentence of reasoning, then a true/false verdict.
If you are uncertain about any criterion, mark it false and explain why in the reasoning.

Respond with ONLY valid JSON. No markdown fences, no preamble, no explanation outside the JSON object.

Output schema:
{{
  "structured_output": {{"reasoning": "...", "verdict": true}},
  "correct_function_selected": {{"reasoning": "...", "verdict": true}},
  "input_format_valid": {{"reasoning": "...", "verdict": true}},
  "no_hallucinated_function": {{"reasoning": "...", "verdict": true}},
  "overall_pass": true
}}
"""

In [57]:
result = openai.chat.completions.create(
    model="gpt-4.1-nano",
    messages=[{"role":"system", "content": judge_llm}])

print(result.choices[0].message.content)

{
  "structured_output": {
    "reasoning": "The response is in the correct format, matching 'FINAL_ANSWER: [number]' with no additional text or markdown.",
    "verdict": true
  },
  "correct_function_selected": {
    "reasoning": "The query explicitly asks for 'fibonacci_numbers of 3,' so calling 'fibonacci_numbers' is correct.",
    "verdict": true
  },
  "input_format_valid": {
    "reasoning": "The function 'fibonacci_numbers' takes an integer as input; here, '3' is given as a number, not a string, which is appropriate.",
    "verdict": true
  },
  "no_hallucinated_function": {
    "reasoning": "The function 'fibonacci_numbers' exists in the system prompt's list, so it is not hallucinated.",
    "verdict": true
  },
  "overall_pass": true
}
